# Creating the Nested Threads

This notebook is for you to work with the CLOY data folder.

<h3>Import requirements</h3>

In [1]:
import json
from pprint import pprint

<h3>Save the submissions</h3>

In [2]:
years = [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [3]:
threads = {}
all_items = {} #create this dict lookup table for comments

for year in years:
    with open(f"data/CLOY/cloy_submissions_{year}.jsonl", "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                item = json.loads(line)
                id = item["post_id"]
                if id:
                    threads[id] = item
                    threads[id]["year"] = year
                    threads[id]["comments"] = {}
                    all_items[id] = item

print(f"number of submissions: {len(threads)}")

number of submissions: 3066


In [4]:
print(list(threads.keys())[0])
pprint(threads["dkxewd"])

dkxewd
{'author': '[deleted]',
 'comments': {},
 'flair': None,
 'full_id': None,
 'num_comments': 0,
 'permalink': 'https://reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
 'post_id': 'dkxewd',
 'score': 1,
 'selftext': '[deleted]',
 'subreddit': 'CrashLandingOnYou',
 'timestamp': 1571646432,
 'title': 'Crash Landing On You has been created',
 'url': 'https://www.reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
 'year': 2019}


<h3>Save the comments (Thank you Mrs. Gemini)</h3>

In [5]:
# 2. First Pass: Load all comments into memory & lookup table
unattached_comments = []

for year in years:
    with open(f"data/CLOY/cloy_comments_{year}.jsonl", "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                comment = json.loads(line)
                comment_id = comment.get("comment_id")
                if comment_id:
                    comment["year"] = year
                    comment["comments"] = {}  # Dictionary for child replies: {comment_id: comment_dict}
                    
                    all_items[comment_id] = comment  # Register in lookup table
                    unattached_comments.append(comment)

# 3. Second Pass: Attach each comment directly to its parent's dict
for comment in unattached_comments:
    comment_id = comment["comment_id"]
    
    # Clean parent_id
    parent_id = str(comment.get("parent_id", ""))
    if parent_id.startswith(("t1_", "t3_")):
        parent_id = parent_id[3:]
    
    # Direct O(1) lookup & assignment using comment_id as the key
    if parent_id in all_items:
        all_items[parent_id]["comments"][comment_id] = comment
    else:
        # Parent post/comment was deleted, missing, or outside dataset scope
        pass

In [8]:
pprint(threads["1hob34z"])

{'author': 'fefeh1',
 'comments': {'m485nfx': {'author': 'No_Olive_3310',
                          'comment_body': 'This was my favorite one too! Their '
                                          'reunion! And RJH was so cute when '
                                          'he was reading the door opening '
                                          'video comments. Episode 12 was '
                                          'great too, the video game kid mode '
                                          'and you can’t beat the soju scene '
                                          '❤️ I loved Episode 13 too, '
                                          'especially the movie theatre '
                                          'armrest scene. Actually, I just '
                                          'love all their SK scenes together. '
                                          'It gives a nice glimpse of what '
                                          'their life would be like if the

<h3>Save nested threads to json file</h3>

In [9]:
with open("data/cloy/nested_threads.json", "w", encoding="utf-8") as file:
    json.dump(threads, file, indent=2)